### Household Rosters
### _hhr.ipynb


Sarah Sullivan

Created: April 7, 2026 

Last Updated: June 8, 2026


This script inputs the file "_psid_long_matrix.dta," outputted by the script _psid.do in part Ia, step 8.

I do some datatype manipulation to create variables measuring changes between household rosters constructed in _psid.do. 

I then output the csv file "_hhr.csv" which gets merged back onto the file "_psid_long.dta" in part X, step X of _psid.do. 

You might be thinking, couldn't I have done this all in Stata? Yeah, I think I could've.

In [ ]:
# import packages
# version of pandas is 2.0.3, numpy is 2.0.3
import numpy as np
import pandas as pd

In [2]:
# read in data output from _psid.do part I step 32. 
output = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/_output"
df = pd.read_stata(output + "/_psid_long_matrix.dta", convert_categoricals=False)

In [4]:
# convert float variables to integers
floatvars = ["fam", "ID", "year", "age_", "fam_id_", "head_rel_", "sample_indiv_N", "sample_indiv_A", "sample_indiv_B", "year_left_survey", "why_left_survey"]

for i in floatvars:
    df.fillna({i:0}, inplace=True)
    df[i] = df[i].astype(int)

In [5]:
df

,fam,ID,year,fam_id_,age_,head_rel_,hhr_matrix,rel_matrix,hhr_no_self,ages_no_self,rel_no_self,sib_list,par_list,gpar_list,sample_indiv_N,sample_indiv_A,sample_indiv_B,why_left_survey,year_left_survey
0,1,1030,1973,17,1,3,1003 1004,30 30,1003 1004,25 23,1 2,,,,1,0,1,1,1980
1,1,1030,1974,377,1,3,1003 1004,30 30,1003 1004,27 25,1 2,,,,1,0,1,1,1980
2,1,1030,1975,2848,3,3,1003 1004,30 30,1003 1004,28 26,1 2,,,,1,0,1,1,1980
3,1,1030,1976,2425,4,3,1003 1004,30 30,1003 1004,30 28,1 2,,,,1,0,1,1,1980
4,1,1030,1977,2012,5,3,1003 1004,30 30,1003 1004,30 28,1 2,,,,1,0,1,1,1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316707,9308,9308003,1995,8660,9,30,9308002 9308004 9308170,30 40 60,9308002 9308004 9308170,28 5 51,10 30 50,9308004,9308002,,1,0,1,80,1996
316708,9308,9308004,1992,9496,3,30,9308002 9308003,30 40,9308002 9308003,25 7,10 30,9308003,9308002,,1,0,1,80,1996
316709,9308,9308004,1993,4839,3,30,9308002 9308003,30 40,9308002 9308003,26 8,10 30,9308003,9308002,,1,0,1,80,1996
316710,9308,9308004,1994,14295,4,30,9308002 9308003,30 40,9308002 9308003,27 9,10 30,9308003,9308002,,1,0,1,80,1996


In [6]:
# convert list-like strings of rosters to real lists
df['hhr_mat'] = [[] for _ in range(len(df))]
df['rel_mat'] = [[] for _ in range(len(df))]
df['hhr'] = [[] for _ in range(len(df))]
df['ages'] = [[] for _ in range(len(df))]
df['rel'] = [[] for _ in range(len(df))]
df['siblings'] = [[] for _ in range(len(df))]
df['parents'] = [[] for _ in range(len(df))]
df['grandparents'] = [[] for _ in range(len(df))]

df['hhr_mat'] = df['hhr_matrix'].apply(lambda x: x.split())
df['rel_mat'] = df['rel_matrix'].apply(lambda x: x.split())

df['hhr'] = df['hhr_no_self'].apply(lambda x: x.split())
df['ages'] = df['ages_no_self'].apply(lambda x: x.split())
df['rel'] = df['rel_no_self'].apply(lambda x: x.split())

df['siblings'] = df['sib_list'].apply(lambda x: x.split())
df['parents'] = df['par_list'].apply(lambda x: x.split())
df['grandparents'] = df['gpar_list'].apply(lambda x: x.split())


In [7]:
# clean her up 
df = df.drop(columns=['hhr_matrix', 'rel_matrix', 'hhr_no_self', 'ages_no_self', 'rel_no_self', 'sib_list', 'par_list', 'gpar_list'])

In [19]:
count_hhr_not_matrix = (df["hhr"] != df["hhr_mat"]).sum()
print(count_hhr_not_matrix)


56


In [ ]:
# generate ages_mat variable, ages of members in household roster matrix constructed as copy of ages of members in household roster
# self-constructued, if they're the same. If not, assign empty list. Empty list for only 56 obs because of non-match and add'l 43 because
# of no non-self age. 

df['ages_mat'] = df.apply(lambda row: row['ages'] if row['hhr_mat'] == row['hhr'] else [], axis=1)

In [17]:
df

,fam,ID,year,fam_id_,age_,head_rel_,sample_indiv_N,sample_indiv_A,sample_indiv_B,why_left_survey,year_left_survey,hhr_mat,rel_mat,hhr,ages,rel,siblings,parents,grandparents,ages_mat
0,1,1030,1973,17,1,3,1,0,1,1,1980,"[1003, 1004]","[30, 30]","[1003, 1004]","[25, 23]","[1, 2]",[],[],[],"[25, 23]"
1,1,1030,1974,377,1,3,1,0,1,1,1980,"[1003, 1004]","[30, 30]","[1003, 1004]","[27, 25]","[1, 2]",[],[],[],"[27, 25]"
2,1,1030,1975,2848,3,3,1,0,1,1,1980,"[1003, 1004]","[30, 30]","[1003, 1004]","[28, 26]","[1, 2]",[],[],[],"[28, 26]"
3,1,1030,1976,2425,4,3,1,0,1,1,1980,"[1003, 1004]","[30, 30]","[1003, 1004]","[30, 28]","[1, 2]",[],[],[],"[30, 28]"
4,1,1030,1977,2012,5,3,1,0,1,1,1980,"[1003, 1004]","[30, 30]","[1003, 1004]","[30, 28]","[1, 2]",[],[],[],"[30, 28]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316707,9308,9308003,1995,8660,9,30,1,0,1,80,1996,"[9308002, 9308004, 9308170]","[30, 40, 60]","[9308002, 9308004, 9308170]","[28, 5, 51]","[10, 30, 50]",[9308004],[9308002],[],"[28, 5, 51]"
316708,9308,9308004,1992,9496,3,30,1,0,1,80,1996,"[9308002, 9308003]","[30, 40]","[9308002, 9308003]","[25, 7]","[10, 30]",[9308003],[9308002],[],"[25, 7]"
316709,9308,9308004,1993,4839,3,30,1,0,1,80,1996,"[9308002, 9308003]","[30, 40]","[9308002, 9308003]","[26, 8]","[10, 30]",[9308003],[9308002],[],"[26, 8]"
316710,9308,9308004,1994,14295,4,30,1,0,1,80,1996,"[9308002, 9308003]","[30, 40]","[9308002, 9308003]","[27, 9]","[10, 30]",[9308003],[9308002],[],"[27, 9]"


In [30]:
# flag observations where hhr and hhr_mat are not the same.
df['flag'] = df.apply(lambda row: row["hhr"] != row["hhr_mat"], axis=1)
df['hhr_diff'] = df.apply(lambda row: set(row["hhr_mat"]).symmetric_difference(set(row["hhr"])), axis=1)

In [ ]:
df['hhr_diff'].value_counts()



hhr_diff
{}                             316656
{5742001}                          38
{1844001}                          12
{2411175}                           3
{938002}                            1
{1290004, 1290001, 1290002}         1
{1290003}                           1
Name: count, dtype: int64

#### Using Matrix observations + own relationship to head + age observations

In [25]:
# within-person previous roster (ordered by year)
df = df.sort_values(["ID", "year"]).copy()

df['hhr_prev'] = df.groupby('ID')['hhr_mat'].shift(1)
df['ages_prev'] = df.groupby('ID')['ages'].shift(1)
df['rel_x_prev'] = df.groupby('ID')['rel_mat'].shift(1)
df['rel_rp_prev'] = df.groupby('ID')['rel'].shift(1)

In [26]:
# fill missings with empty lists
df['hhr_prev'] = df['hhr_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['ages_prev'] = df['ages_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['rel_x_prev'] = df['rel_x_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['rel_rp_prev'] = df['rel_rp_prev'].apply(lambda d: d if isinstance(d, list) else [])

In [27]:
# ids of who left and who came in each year (within person)
df["IDs_left"] = df.apply(
    lambda row: [x for x in row["hhr_prev"] if x not in row["hhr_mat"]],
    axis=1
)
df["IDs_came"] = df.apply(
    lambda row: [x for x in row["hhr_mat"] if x not in row["hhr_prev"]],
    axis=1
)

In [28]:
# flag first and last year of observation for each individual
df['is_first'] = ~df['ID'].duplicated(keep='first')
df['is_last'] = ~df['ID'].duplicated(keep='last')

In [29]:
df['IDs_came'] = df.apply(lambda row: [] if row['is_first'] else row['IDs_came'], axis=1)
df['IDs_left'] = df.apply(lambda row: [] if row['is_last'] else row['IDs_left'], axis=1)

In [15]:
# function for age of who left/came
def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [ages[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(ages)]  

In [16]:
# function for relationship of who left/came to head
def get_rel_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    rel_prev = row['rel_prev'] if isinstance(row['rel_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [rel_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(rel_prev)]

def get_rel_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    rel = row['rel'] if isinstance(row['rel'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [rel[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(rel)]  

In [17]:
# function for relationship of who left/came to INDIVIDUAL - Matrix version
def get_rel_left_mat(row):
    hhr_prev_mat = row['hhr_mat_prev'] if isinstance(row['hhr_mat_prev'], list) else []
    rel_prev_mat = row['rel_mat_prev'] if isinstance(row['rel_mat_prev'], list) else []
    IDs_left_mat = row['IDs_left_mat'] if isinstance(row['IDs_left_mat'], list) else []
    return [rel_prev_mat[hhr_prev_mat.index(pid)] for pid in IDs_left_mat if pid in hhr_prev_mat and hhr_prev_mat.index(pid) < len(rel_prev_mat)]

def get_rel_came_mat(row):
    hhr_mat = row['hhr_mat'] if isinstance(row['hhr_mat'], list) else []
    rel_mat = row['rel_mat'] if isinstance(row['rel_mat'], list) else []
    IDs_came_mat = row['IDs_came_mat'] if isinstance(row['IDs_came_mat'], list) else []
    return [rel_mat[hhr_mat.index(pid)] for pid in IDs_came_mat if pid in hhr_mat and hhr_mat.index(pid) < len(rel_mat)]  

In [18]:
# apply fns above 

df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

df['rel_left'] = df.apply(get_rel_left, axis=1)
df['rel_came'] = df.apply(get_rel_came, axis=1)

df['rel_left_mat'] = df.apply(get_rel_left_mat, axis=1)
df['rel_came_mat'] = df.apply(get_rel_came_mat, axis=1)

In [19]:
# convert elements of ages_left and ages_came to list of integers
df['ages_left'] = df['ages_left'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])
df['ages_came'] = df['ages_came'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])

In [20]:
df

,fam,ID,year,fam_id_,age_,head_rel_,sample_indiv_N,sample_indiv_A,sample_indiv_B,why_left_survey,...,IDs_left_mat,IDs_came_mat,is_first,is_last,ages_left,ages_came,rel_left,rel_came,rel_left_mat,rel_came_mat
0,1,1030,1973,17,1,3,1,0,1,1,...,[],[],True,False,[],[],[],[],[],[]
1,1,1030,1974,377,1,3,1,0,1,1,...,[],[],False,False,[],[],[],[],[],[]
2,1,1030,1975,2848,3,3,1,0,1,1,...,[],[],False,False,[],[],[],[],[],[]
3,1,1030,1976,2425,4,3,1,0,1,1,...,[],[],False,False,[],[],[],[],[],[]
4,1,1030,1977,2012,5,3,1,0,1,1,...,[],[],False,False,[],[],[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316707,9308,9308003,1995,8660,9,30,1,0,1,80,...,[],[],False,True,[],[51],[],[50],[],[]
316708,9308,9308004,1992,9496,3,30,1,0,1,80,...,[],[],True,False,[],[],[],[],[],[]
316709,9308,9308004,1993,4839,3,30,1,0,1,80,...,[],[],False,False,[],[],[],[],[],[]
316710,9308,9308004,1994,14295,4,30,1,0,1,80,...,[],[],False,False,[],[],[],[],[],[]


Stopped june 8th 10pm --> create variables here for relative vs. non-relative came/left, think about ages. 

In [21]:
# sort by adult vs. child came or left. 

df['adult_came'] = df['ages_came'].apply(
    lambda ages: isinstance(ages, list) and any(age >= 18 for age in ages)
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: isinstance(ages, list) and any(age < 18 for age in ages)
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: isinstance(ages, list) and any(age >= 18 for age in ages)
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: isinstance(ages, list) and any(age < 18 for age in ages)
)

In [22]:
df['sib_came'] = df.apply(
    lambda row: (isinstance(row['IDs_came'], list) and isinstance(row['siblings'], list) and any(id in row['siblings'] for id in row['IDs_came'])), axis=1
    )

df['sib_left'] = df.apply(
    lambda row: (isinstance(row['IDs_left'], list) and isinstance(row['siblings'], list) and any(id in row['siblings'] for id in row['IDs_left'])), axis=1
)   

In [23]:
# Get ages of siblings who came
def get_sib_ages_came(row):
    if not isinstance(row['IDs_came'], list) or not isinstance(row['siblings'], list):
        return []
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages_hhr'] if isinstance(row['ages_hhr'], list) else []
    # Get IDs that are both in IDs_came and siblings
    sibs_IDs_came = [id for id in row['IDs_came'] if id in row['siblings']]
    # Get ages for those siblings
    return [ages[hhr.index(sib_id)] for sib_id in sibs_IDs_came if sib_id in hhr and hhr.index(sib_id) < len(ages)]

# Get ages of siblings who left
def get_sib_ages_left(row):
    if not isinstance(row['IDs_left'], list) or not isinstance(row['siblings'], list):
        return []
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    # Get IDs that are both in IDs_left and siblings
    sibs_IDs_left = [id for id in row['IDs_left'] if id in row['siblings']]
    # Get ages for those siblings
    return [ages_prev[hhr_prev.index(sib_id)] for sib_id in sibs_IDs_left if sib_id in hhr_prev and hhr_prev.index(sib_id) < len(ages_prev)]

df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)

KeyError: 'ages_hhr'

In [ ]:
df['par_came'] = df.apply(
    lambda row: int(isinstance(row['IDs_came'], list) and isinstance(row['parents'], list) and any(id in row['parents'] for id in row['IDs_came'])),
    axis=1
)

df['par_left'] = df.apply(
    lambda row: int(isinstance(row['IDs_left'], list) and isinstance(row['parents'], list) and any(id in row['parents'] for id in row['IDs_left'])),
    axis=1
)

In [ ]:
df['gpar_came'] = df.apply(
    lambda row: int(isinstance(row['IDs_came'], list) and isinstance(row['grandparents'], list) and any(id in row['grandparents'] for id in row['IDs_came'])),
    axis=1
)

df['gpar_left'] = df.apply(
    lambda row: int(isinstance(row['IDs_left'], list) and isinstance(row['grandparents'], list) and any(id in row['grandparents'] for id in row['IDs_left'])),
    axis=1
)

In [ ]:
# flags for changes, in, and out. 
df['hhr_change'] = df.apply(lambda row: 1 if (len(row['IDs_came']) > 0) or (len(row['IDs_left']) > 0) else 0, axis=1)
df['hhr_in'] = df.apply(lambda row: 1 if len(row['IDs_came']) > 0 else 0, axis=1)
df['hhr_out'] = df.apply(lambda row: 1 if len(row['IDs_left']) > 0 else 0, axis=1)

In [ ]:
# output
df.to_csv(f"{output}/_hhr.csv", index=False)